# 05: Embedding Paradigms

Builds and compares the three Section 3.4 embedding paradigms (FT-Transformer, SubTab, SCARF) on top of the already-fitted encoders from `04_feature_encoding.ipynb`.

## Setup

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import config

FINAL_DIR = config.FINAL_DIR

sns.set_theme(style="whitegrid")

from src.features.target_split import build_feature_target_split
from src.features.pipeline import FittedEncoders

## 1. Load active candidate

In [ ]:
CANDIDATES = {
    "br_gt0": {
        "train": FINAL_DIR / "tmdb_br_gt0_train.parquet",
        "val": FINAL_DIR / "tmdb_br_gt0_val.parquet",
        "test": FINAL_DIR / "tmdb_br_gt0_test.parquet",
    },
    "nbr_gt5": {
        "train": FINAL_DIR / "tmdb_nbr_gt5_train.parquet",
        "val": FINAL_DIR / "tmdb_nbr_gt5_val.parquet",
        "test": FINAL_DIR / "tmdb_nbr_gt5_test.parquet",
    },
}

In [ ]:
# Single switch: "br_gt0" is the primary candidate, "nbr_gt5" the
# fallback. Every later cell reads train_df/val_df/test_df (or
# ACTIVE_CANDIDATE for naming saved artifacts) instead of a literal
# candidate name, so switching candidates means editing only this line.
ACTIVE_CANDIDATE = "br_gt0"
# ACTIVE_CANDIDATE = "nbr_gt5"

In [ ]:
train_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["train"])
val_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["val"])
test_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["test"])

FINAL_SPLITS = {"train": train_df, "val": val_df, "test": test_df}

train_split = build_feature_target_split(train_df)

ARTIFACTS_DIR = REPO_ROOT / "data" / "artifacts" / ACTIVE_CANDIDATE
fitted_encoders = FittedEncoders.fit(FINAL_SPLITS, train_split, ARTIFACTS_DIR)

print(f"Active candidate: {ACTIVE_CANDIDATE}")
print(f"  train: {train_df.shape}")
print(f"  val:   {val_df.shape}")
print(f"  test:  {test_df.shape}")
print(f"fitted_encoders.dims: {fitted_encoders.dims}")

## 2. Paradigm hyperparameter configs

Instantiate all four `src/models/paradigm_config.py` dataclasses and print every field before anything trains on them -- a quick eyeball check against the papers they're cited from.

In [ ]:
import dataclasses

from src.models.paradigm_config import (
    FTTransformerConfig,
    SubTabConfig,
    SCARFConfig,
    SharedTrainingConfig,
)

ft_transformer_config = FTTransformerConfig()
subtab_config = SubTabConfig()
scarf_config = SCARFConfig()
shared_training_config = SharedTrainingConfig()

for name, cfg in [
    ("FTTransformerConfig", ft_transformer_config),
    ("SubTabConfig", subtab_config),
    ("SCARFConfig", scarf_config),
    ("SharedTrainingConfig", shared_training_config),
]:
    print(f"--- {name} ---")
    for key, value in dataclasses.asdict(cfg).items():
        print(f"  {key}: {value!r}")
    print()